### Prerequisites

In [ ]:
import anndata

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.stats import median_abs_deviation

import celltypist
from celltypist import models
import matplotlib.pyplot as plt

import os

import scvi
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)

sc.settings.verbosity = 3
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white', frameon=False)

### Tidy up data

In [ ]:
import os,glob,tarfile,scanpy as sc
from scipy import sparse as sp
input_dir='../yourpath/raw_data/GSE155698_RAW/'
output_dir='../yourpath/1_tidyup/'

def load_10X_export_mtx(input_directory,output_directory):
    os.makedirs(output_directory,exist_ok=True)
    for filename in os.listdir(input_directory):
        if not filename.endswith('.tar.gz'): continue
        sample=filename.replace('.tar.gz','')
        tar_path=os.path.join(input_directory,filename)
        extract_path=os.path.join(input_directory,sample+'_mtx')
        print(f'⏳ Processing sample: {sample}')
        os.makedirs(extract_path,exist_ok=True)
        with tarfile.open(tar_path,'r:gz') as tar: tar.extractall(path=extract_path)
        candidates=glob.glob(os.path.join(extract_path,'**','filtered_feature_bc_matrix'),recursive=True)
        if not candidates: raise FileNotFoundError(f"Could not find 'filtered_feature_bc_matrix' under {extract_path}")
        mtx_dir=candidates[0]
        adata=sc.read_10x_mtx(mtx_dir,var_names='gene_symbols',make_unique=True)
        adata.obs_names_make_unique(); adata.var_names_make_unique()
        sc.pp.filter_cells(adata,min_genes=200); sc.pp.filter_genes(adata,min_cells=1)
        adata.X=sp.csr_matrix(adata.X); adata.obs['sample']=sample
        out_file=os.path.join(output_directory,sample+'.h5ad')
        print(f'✅ Writing {out_file}'); adata.write(out_file)
    print('All samples processed.')
load_10X_export_mtx(input_dir,output_dir)


### QC

In [ ]:
#Variables
FOLDER_PATH_KEY = '../yourpath/1_tidyup/'
FILENAME_STRING_END = '.h5ad'

ribo_url = '../20240211_broadinstitute_KEGG_RIBOSOME.v2023.2.Hs.txt'
ribo_genes = pd.read_table(ribo_url, skiprows=2, header = None)

#Functions
def read_file_to_adata(csv_file):    
    adata = sc.read_h5ad(filename=csv_file)
    adata.var_names_make_unique()

    return adata

def is_outlier(adata, metric: str, nmads: int):
    M = adata.obs[metric]
    outlier = (M < np.median(M) - nmads * median_abs_deviation(M)) | (
        np.median(M) + nmads * median_abs_deviation(M) < M
    )
    return outlier

def preprocess_directory_to_adata():
    folder_path = FOLDER_PATH_KEY
    out = []
    adata = None
    
    for file_name in os.listdir(folder_path):
        if file_name.endswith(FILENAME_STRING_END):
            file_path = os.path.join(folder_path, file_name)
            adata = read_file_to_adata(file_path)

            #Detect doublets
            sc.pp.filter_genes(adata, min_cells = 10)
            sc.pp.filter_genes(adata, max_counts = 100000)
            sc.pp.filter_cells(adata, min_genes = 10)
            sc.pp.highly_variable_genes(adata, n_top_genes = 2000, subset = True, flavor = 'seurat_v3')

            scvi.model.SCVI.setup_anndata(adata)
            vae = scvi.model.SCVI(adata)
            vae.train(accelerator="cpu", devices=1)
            solo = scvi.external.SOLO.from_scvi_model(vae)
            solo.train(accelerator="cpu", devices=1)

            df = solo.predict()
            df['prediction'] = solo.predict(soft = False)
            df['dif'] = df.doublet - df.singlet
            doublets = df[(df.prediction == 'doublet') & (df.dif > 1)]
            print(f"Total number of potential doublets: {len(df['dif'])}")
            print(f"Total number of doublets: {len(doublets['prediction'])}")

            adata = read_file_to_adata(file_path)
            sample_name = file_name
            adata.obs['filename'] = sample_name
            print(f"Total number of cells: {adata.n_obs}")
            adata.obs['doublet'] = adata.obs.index.isin(doublets.index)
            adata = adata[~adata.obs.doublet]
            print(f"Number of cells after filtering of doublets: {adata.n_obs}")

            #Remove cells based on mitochondrial genes and total genes
            sc.pp.filter_cells(adata, min_genes = 200) 

            adata.var['mt'] = adata.var_names.str.startswith('MT-')
            adata.var['ribo'] = adata.var_names.isin(ribo_genes[0].values)
            adata.var["hb"] = adata.var_names.str.contains(("^HB[^(P)]"))
            sc.pp.calculate_qc_metrics(adata, qc_vars=['mt', 'ribo', 'hb'], percent_top=[20], log1p=True, inplace=True)

            adata.obs["outlier"] = (
            is_outlier(adata, "log1p_total_counts", 5)
            | is_outlier(adata, "log1p_n_genes_by_counts", 5)
            | is_outlier(adata, "pct_counts_in_top_20_genes", 5)
            )
            adata.obs.outlier.value_counts()
            adata.obs["mt_outlier"] = is_outlier(adata, "pct_counts_mt", 3) | (
            adata.obs["pct_counts_mt"] > 20
            )
            adata.obs.mt_outlier.value_counts()
            print(f"Total number of cells: {adata.n_obs}")
            adata = adata[(~adata.obs.outlier) & (~adata.obs.mt_outlier)].copy()
            print(f"Number of cells after filtering of low quality cells: {adata.n_obs}")

            out.append(adata)
    
    if out:
        adata = sc.concat(out, join='outer')
    return adata

adata = preprocess_directory_to_adata()
adata


In [ ]:
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt', color='filename')
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', color='filename')

In [4]:
adata.write('2_QC.h5ad')

### Annotation and metadata

In [5]:
#Load data
adata = sc.read_h5ad('2_QC.h5ad')

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.highly_variable_genes(adata, flavor="seurat_v3", n_top_genes=6000, layer="counts", subset=True)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata

scvi.model.SCVI.setup_anndata(adata, layer='counts',
                              batch_key='filename',
                              continuous_covariate_keys=['pct_counts_mt', 'total_counts', 'pct_counts_ribo'])

model = scvi.model.SCVI(adata, n_layers=2, n_latent=30)
model.train(accelerator="cpu", devices=1)
SCVI_LATENT_KEY = "X_scVI"
adata.obsm[SCVI_LATENT_KEY] = model.get_latent_representation()
sc.pp.neighbors(adata, use_rep='X_scVI')
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=1)
adata.layers['scVI_normalized'] = model.get_normalized_expression(library_size= 1e4)

In [ ]:
#Add annotation 2
cell_type = {
    '0': 'Monocytes',
    '1': 'Monocytes',
    '2': 'T/NK cells',
    '3': 'T/NK cells',
    '4': 'Monocytes (classical)',
    '5': 'Epithelial cells',
    '6': 'T/NK cells',
    '7': 'T/NK cells',
    '8': 'Macrophages',
    '9': 'Epithelial cells',
    '10': 'Macrophages',
    '11': 'T/NK cells',
    '12': 'T/NK cells',
    '13': 'Monocytes',
    '14': 'Monocytes',
    '15': 'T/NK cells',
    '16': 'Monocytes',
    '17': 'Stromal cells',
    '18': 'B cells',
    '19': 'Doublets',
    '20': 'Mast cells',
    '21': 'Monocytes (non-classical)',
    '22': 'Plasma cells',
    '23': 'Epithelial cells',
    '24': 'Stromal cells',
    '25': 'T/NK cells',
    '26': 'T/NK cells',
    '27': 'Stromal cells',
    '28': 'Macrophages',
    '29': 'Endothelial cells',
    '30': 'Epithelial cells',
    '31': 'pDC',
    '32': 'Unknown',
    '33': 'Macrophages',
    '34': 'Doublets',
    '35': 'Neutrophils',
   
}
adata.obs['anno'] = adata.obs.leiden.map(cell_type)
adata_anno = adata.obs['anno']
bdata = sc.read_h5ad(filename='2_QC.h5ad')
bdata.obs['anno'] = adata_anno
bdata.obs['anno2'] = adata.obs['anno']

In [ ]:
sc.pl.heatmap(adata, anno_genes, groupby='anno', swap_axes=True, cmap='Greys', vmax="5",show_gene_labels=True)

In [ ]:
#Add metadata
metadata = pd.read_excel('../yourpath/raw_data/metadata.xlsx')
merged_df = bdata.obs.merge(metadata, how="left", on="filename")
bdata.obs = merged_df
bdata.obs.index = adata.obs.index
bdata.obs

In [73]:
bdata.obs['patient'] = bdata.obs['patient'].astype(str)
bdata.obsm['X_scVI'] = adata.obsm['X_scVI']
bdata.write('3_metadata.h5ad')


... storing 'patient' as categorical


In [ ]:
cluster_tissue_matrix = pd.crosstab(bdata.obs['anno'], bdata.obs['tissue'])
cluster_tissue_matrix